In [0]:
from pyspark.sql import functions as f
from pyspark.sql import Window


print("-----GIVEN DATAFRAME-----\n\n\n")


data = [(1111, "2021-01-15", 10),
        (1111, "2021-01-16", 15),
        (1111, "2021-01-17", 30),
        (1112, "2021-01-15", 10),
        (1112, "2021-01-15", 20),
        (1112, "2021-01-15", 30)]

myschema = ["sensorid", "timestamp", "values"]

df = spark.createDataFrame(data, schema=myschema)
display(df)

print("-----REQUIRED DATAFRAME-----\n\n\n")

data1 = [
    (1111, "2021-01-16", 5),
    (1111, "2021-01-17", 15),
    (1112, "2021-01-15", 10),
    (1112, "2021-01-15", 10)
]

df1 = spark.createDataFrame(data1, ["sensorid", "timestamp", "values"])
df1.show()



print("-----DSL SOLUTION-----\n\n\n")


windowdf = Window.partitionBy("sensorid").orderBy("timestamp")

partitiondf = df.withColumn("prevalues", f.lag("values", 1).over(windowdf))
display(partitiondf)

finaldf = partitiondf.withColumn("diff", f.col("values") - f.col("prevalues")) \
    .drop("values", "prevalues") \
    .withColumnRenamed("diff", "values") \
    .filter(f.col("values").isNotNull())

finaldf.show()


print("-----SQL SOLUTION-----\n\n\n")

df.createTempView("sqldfm1")

spark.sql("""
          select * from
          (select sensorid, timestamp, values - (lag(values,1) over (partition by sensorid order by timestamp)) as values
          from sqldfm1)
          where values is not null""").show()


